## Setup — SparkSession and data load

In [2]:
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType, IntegerType

spark = (
    SparkSession.builder.appName("FlightDelayAssignment2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

CSV_PATH = Path.cwd() / "Flight Dataset - CSV(in).csv"

raw_df = (
    spark.read.option("header", True)
    .option("inferSchema", False)
    .csv(str(CSV_PATH))
)

flights = (
    raw_df.withColumn("FL_DATE", F.to_date(F.col("FL_DATE"), "M/d/yyyy").cast(DateType()))
    .withColumn("DEP_DELAY", F.col("DEP_DELAY").cast(DoubleType()))
    .withColumn("ARR_DELAY", F.col("ARR_DELAY").cast(DoubleType()))
    .withColumn("AIR_TIME", F.col("AIR_TIME").cast(DoubleType()))
    .withColumn("DISTANCE", F.col("DISTANCE").cast(DoubleType()))
    .withColumn("DEP_TIME", F.col("DEP_TIME").cast(DoubleType()))
    .withColumn("ARR_TIME", F.col("ARR_TIME").cast(DoubleType()))
).cache()

flights.printSchema()
print(f"Row count: {flights.count():,}")
flights.show(3, truncate=False)

c:\Kanhaiya\assignment\mini-assignments\.venv-1\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- FL_DATE: date (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)

Row count: 1,000,000
+----------+---------+---------+--------+--------+---------+---------+
|FL_DATE   |DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE|DEP_TIME |ARR_TIME |
+----------+---------+---------+--------+--------+---------+---------+
|2006-01-01|5.0      |19.0     |350.0   |2475.0  |9.083333 |12.483334|
|2006-01-02|167.0    |216.0    |343.0   |2475.0  |11.783334|15.766666|
|2006-01-03|-7.0     |-2.0     |344.0   |2475.0  |8.883333 |12.133333|
+----------+---------+---------+--------+--------+---------+---------+
only showing top 3 rows


## Task 1 — Number of flights that arrived earlier than expected

In [3]:
def flights_arrived_early(df: DataFrame) -> int:
    return df.filter(F.col("ARR_DELAY") < 0).count()


early_count = flights_arrived_early(flights)
print(f"Flights that arrived earlier than expected: {early_count:,}")

Flights that arrived earlier than expected: 534,655


## Task 2 — Typical departure time for flights over 2000 miles

In [4]:
def _hours_to_hhmm(hours: float) -> str:
    total_minutes = int(round(hours * 60))
    hh, mm = divmod(total_minutes, 60)
    return f"{hh:02d}:{mm:02d}"


def typical_dep_time_long_haul(df: DataFrame, distance_threshold: float = 2000.0) -> dict:
    long_haul = df.filter(F.col("DISTANCE") > distance_threshold)
    stats = long_haul.agg(
        F.count("*").alias("n_flights"),
        F.avg("DEP_TIME").alias("mean_dep_time"),
        F.expr("percentile_approx(DEP_TIME, 0.5)").alias("median_dep_time"),
    ).first()
    return {
        "n_flights_over_2000mi": int(stats["n_flights"]),
        "mean_dep_time_hours": float(stats["mean_dep_time"]),
        "mean_dep_time_hhmm": _hours_to_hhmm(float(stats["mean_dep_time"])),
        "median_dep_time_hours": float(stats["median_dep_time"]),
        "median_dep_time_hhmm": _hours_to_hhmm(float(stats["median_dep_time"])),
    }


dep_stats = typical_dep_time_long_haul(flights)
for k, v in dep_stats.items():
    print(f"  {k}: {v}")

  n_flights_over_2000mi: 46993
  mean_dep_time_hours: 13.973233947624635
  mean_dep_time_hhmm: 13:58
  median_dep_time_hours: 13.6
  median_dep_time_hhmm: 13:36


## Task 3 — Proportion of flights with arrival delay longer than 60 minutes

In [4]:
def proportion_arr_delay_gt_60(df: DataFrame) -> float:
    stats = df.agg(
        F.count("ARR_DELAY").alias("n_with_arr_delay"),
        F.sum(F.when(F.col("ARR_DELAY") > 60, 1).otherwise(0)).alias("n_gt_60"),
    ).first()
    return float(stats["n_gt_60"]) / float(stats["n_with_arr_delay"])


proportion = proportion_arr_delay_gt_60(flights)
print(f"Proportion of flights with ARR_DELAY > 60 min: {proportion:.6f} ({proportion * 100:.2f}%)")

Proportion of flights with ARR_DELAY > 60 min: 0.053066 (5.31%)


## Task 4 — Average airtime for flights that left before 9:00 AM

In [5]:
def avg_airtime_before_9am(df: DataFrame) -> float:
    return float(
        df.filter(F.col("DEP_TIME") < 9.0)
        .agg(F.avg("AIR_TIME").alias("avg_airtime"))
        .first()["avg_airtime"]
    )


avg_air = avg_airtime_before_9am(flights)
print(f"Average AIR_TIME for flights departing before 09:00: {avg_air:.2f} minutes")

Average AIR_TIME for flights departing before 09:00: 111.36 minutes


## Task 5 — Max arrival delay for flights that did not experience a departure delay

In [6]:
def max_arr_delay_no_dep_delay(df: DataFrame) -> float:
    return float(
        df.filter(F.col("DEP_DELAY") <= 0)
        .agg(F.max("ARR_DELAY").alias("max_arr_delay"))
        .first()["max_arr_delay"]
    )


max_arr = max_arr_delay_no_dep_delay(flights)
print(f"Max ARR_DELAY when DEP_DELAY <= 0: {max_arr:.0f} minutes")

Max ARR_DELAY when DEP_DELAY <= 0: 701 minutes


In [7]:
spark.stop()